<a href="https://colab.research.google.com/github/chaimaao0/ml-data-science-internship/blob/main/NLP_Sentiment_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Import Libraries

In [26]:
import re
import glob
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

True

## 2. Load the Data

In [27]:
import os

df = pd.read_csv("/content/sample_data/sentiment_reviews.csv")
print("Shape:", df.shape)
df.head()

Shape: (10, 3)


,review_id,review_text,sentiment
0,1,"The product quality is exceptional, exceeded a...",positive
1,2,Terrible customer service and the item arrived...,negative
2,3,"Average performance, decent for the price poin...",neutral
3,4,"Absolutely love this! Easy to setup, robust, a...",positive
4,5,Stopped working after two days. Extremely disa...,negative


In [28]:
print("Missing values:", df.isnull().sum().sum())
print()
print(df["sentiment"].value_counts())

Missing values: 0

sentiment
positive    5
negative    4
neutral     1
Name: count, dtype: int64


In [29]:
df = df[df["sentiment"].isin(["positive", "negative"])].reset_index(drop=True)

df["label"] = (df["sentiment"] == "positive").astype(int)

print("Reviews kept:", len(df))
print(df["sentiment"].value_counts())

Reviews kept: 9
sentiment
positive    5
negative    4
Name: count, dtype: int64


## 3. Preprocess the Text

In [30]:
stop_words = set(stopwords.words("english")) - {"not", "no", "nor"}
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"n't\b", " not", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    words = [lemmatizer.lemmatize(w, pos="v") for w in words]
    return " ".join(words)

In [31]:
df["clean_text"] = df["review_text"].apply(clean_text)

for i in range(3):
    print("Original:", df.loc[i, "review_text"])
    print("Cleaned :", df.loc[i, "clean_text"])
    print()

Original: The product quality is exceptional, exceeded all my expectations!
Cleaned : product quality exceptional exceed expectation

Original: Terrible customer service and the item arrived completely damaged.
Cleaned : terrible customer service item arrive completely damage

Original: Absolutely love this! Easy to setup, robust, and works flawlessly.
Cleaned : absolutely love easy setup robust work flawlessly



## 4. Split and TF-IDF

In [32]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"])

print("Train:", len(X_train_text), "| Test:", len(X_test_text))

Train: 7 | Test: 2


In [33]:
tfidf = TfidfVectorizer(max_features=5000)

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print("TF-IDF matrix (train):", X_train.shape)
print("Some words in the vocabulary:", list(tfidf.get_feature_names_out()[:10]))

TF-IDF matrix (train): (7, 45)
Some words in the vocabulary: ['absolutely', 'alternative', 'arrive', 'available', 'better', 'build', 'clean', 'completely', 'control', 'customer']


## 5. Train the Models

In [34]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

models = {"Multinomial Naive Bayes": nb_model, "Logistic Regression": lr_model}

## 6. Accuracy Benchmarks

In [35]:
results = {}
for name, model in models.items():
    pred = model.predict(X_test)
    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
    }

results = pd.DataFrame(results).T
print("Number of test reviews:", len(y_test))
results.round(4)

Number of test reviews: 2


,Accuracy,Precision,Recall,F1-score
Multinomial Naive Bayes,0.5,0.5,1.0,0.6667
Logistic Regression,0.5,0.5,1.0,0.6667


In [36]:
for name, model in models.items():
    print("===", name, "===")
    print(classification_report(y_test, model.predict(X_test),
                                labels=[0, 1], target_names=["negative", "positive"], zero_division=0))

=== Multinomial Naive Bayes ===
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         1
    positive       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2

=== Logistic Regression ===
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         1
    positive       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



In [37]:

n_splits = int(min(5, df["label"].value_counts().min()))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

print("Cross-validation folds:", n_splits)
for name, model in [("Multinomial Naive Bayes", MultinomialNB()),
                    ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42))]:
    pipe = make_pipeline(TfidfVectorizer(max_features=5000), model)
    scores = cross_val_score(pipe, df["clean_text"], df["label"], cv=cv, scoring="accuracy")
    print(f"{name:<25} CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

Cross-validation folds: 4
Multinomial Naive Bayes   CV accuracy: 0.4583 (+/- 0.0722)
Logistic Regression       CV accuracy: 0.4583 (+/- 0.0722)


### Leave-one-out evaluation (more honest for a tiny dataset)

With only 9 labelled reviews, the 80/20 split above leaves just **2 test reviews**, so a single mistake moves accuracy by 50 points. Leave-one-out cross-validation trains on 8 reviews, predicts the 9th, and repeats for every review, so **all 9 reviews are used as a test case exactly once**. Treat this as the more trustworthy benchmark, but still a rough one: 9 samples cannot give a stable estimate.

In [38]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict

loo_results = {}
loo_predictions = {}

for name, model in [("Multinomial Naive Bayes", MultinomialNB()),
                    ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42))]:
    pipe = make_pipeline(TfidfVectorizer(max_features=5000), model)
    pred = cross_val_predict(pipe, df["clean_text"], df["label"], cv=LeaveOneOut())
    loo_predictions[name] = pred
    loo_results[name] = {
        "Accuracy": accuracy_score(df["label"], pred),
        "Precision": precision_score(df["label"], pred, zero_division=0),
        "Recall": recall_score(df["label"], pred, zero_division=0),
        "F1-score": f1_score(df["label"], pred, zero_division=0),
    }

print("Leave-one-out benchmark (9 predictions per model)")
pd.DataFrame(loo_results).T.round(4)

Leave-one-out benchmark (9 predictions per model)


,Accuracy,Precision,Recall,F1-score
Multinomial Naive Bayes,0.1111,0.2,0.2,0.2
Logistic Regression,0.1111,0.2,0.2,0.2


In [39]:
errors = df[["review_text", "sentiment"]].copy()
for name, pred in loo_predictions.items():
    errors[name] = np.where(pred == 1, "positive", "negative")

errors["NB correct"] = errors["Multinomial Naive Bayes"] == errors["sentiment"]
errors["LR correct"] = errors["Logistic Regression"] == errors["sentiment"]
errors[["review_text", "sentiment", "Multinomial Naive Bayes", "Logistic Regression"]]

,review_text,sentiment,Multinomial Naive Bayes,Logistic Regression
0,"The product quality is exceptional, exceeded a...",positive,negative,negative
1,Terrible customer service and the item arrived...,negative,positive,positive
2,"Absolutely love this! Easy to setup, robust, a...",positive,negative,negative
3,Stopped working after two days. Extremely disa...,negative,positive,positive
4,Customer support helped resolve my setup quest...,positive,positive,positive
5,Delivery was slow and package was missing one ...,negative,positive,positive
6,Clean interface and intuitive controls. Highly...,positive,negative,negative
7,"Not worth the money, plenty of better alternat...",negative,positive,positive
8,Great value and durable build. Would definitel...,positive,negative,negative


## 7. Test Your Own Sentences

In [40]:
def predict_sentiment(text):
    """Predict if a sentence is positive or negative, with both models."""
    vector = tfidf.transform([clean_text(text)])

    print("Text:", text)
    if vector.nnz == 0:
        print("  None of the words are in the training vocabulary, so the models cannot decide.\n")
        return

    for name, model in models.items():
        probabilities = model.predict_proba(vector)[0]
        label = "positive" if probabilities[1] > probabilities[0] else "negative"
        print(f"  {name:<25} -> {label} ({max(probabilities):.0%} confidence)")
    print()

In [41]:

predict_sentiment("This product is great, I love it!")
predict_sentiment("Terrible quality, it stopped working after two days.")
predict_sentiment("It does not work, not worth the money.")
predict_sentiment("Easy to use and very durable.")

Text: This product is great, I love it!
  Multinomial Naive Bayes   -> positive (69% confidence)
  Logistic Regression       -> positive (63% confidence)

Text: Terrible quality, it stopped working after two days.
  Multinomial Naive Bayes   -> negative (55% confidence)
  Logistic Regression       -> positive (51% confidence)

Text: It does not work, not worth the money.
  Multinomial Naive Bayes   -> negative (57% confidence)
  Logistic Regression       -> positive (50% confidence)

Text: Easy to use and very durable.
  Multinomial Naive Bayes   -> positive (67% confidence)
  Logistic Regression       -> positive (62% confidence)



In [42]:
RUN_INTERACTIVE = True

while RUN_INTERACTIVE:
    phrase = input("Type a review (or 'quit' to stop): ")
    if phrase.strip().lower() == "quit":
        break
    predict_sentiment(phrase)

Type a review (or 'quit' to stop): Absolutely love it
Text: Absolutely love it
  Multinomial Naive Bayes   -> positive (67% confidence)
  Logistic Regression       -> positive (62% confidence)

Type a review (or 'quit' to stop): quit


## 8. Conclusion

**What was built:** a full text-classification pipeline: HTML/punctuation removal, lowercasing, stopword removal (keeping *not / no / nor* so negation survives), lemmatization, TF-IDF features (`max_features=5000`), and two classifiers (Multinomial Naive Bayes and Logistic Regression) with a `predict_sentiment()` helper for live phrases.

**What the benchmarks say:** the numbers in this notebook should be read as a smoke test, not as proof of model quality.
- The provided file has only 10 reviews. One is neutral and was dropped, leaving 9 (5 positive, 4 negative), and the vocabulary is only about 45 words.
- The 80/20 split gives only 2 test reviews, which is why accuracy lands at 50% for both models, and cross-validation stays below chance (about 46%). Words that appear in a held-out review are often absent from the 7-8 training reviews, so the models have nothing to go on.
- Leave-one-out (section 6) uses every review as a test case once, so it is a fairer comparison than the 2-review split. Even so, with only 9 samples the estimate remains unreliable, which shows that the limitation is the data size, not the code.

**Model behaviour:** in our tests, Naive Bayes did better on the custom phrases (4/4 vs 2/4), but with so few samples this is not conclusive. Logistic Regression is more regularised by default and, with so few examples, stays close to 50/50 (it called "It does not work, not worth the money" positive at 50%). With a realistic dataset of thousands of reviews (IMDB, Amazon, Yelp), Logistic Regression usually matches or beats Naive Bayes.
